In [ ]:
from pathlib import Path
import polars as pl

In [ ]:
# Assuming either all *_compressed.tar.zst files or all_groups_lean.tar.zst file is/are decompressed
CWD = Path().resolve()
BASE = CWD.parent

print(f"Folder where group folder should be: {BASE}")
print(f"Current working directory (where table will be saved): {CWD}")

In [ ]:
groups = [
    "fungi_mit",
    "metazoans_mit",
    "plants_mit",
    "plants_plt",
    "protists_mit",
    "protists_plt",
    "green_algae_mit",
    "green_algae_plt",
]

label_map = {
    "fungi_mit": "Fungi (mitochondria)",
    "green_algae_mit": "Green algae (mitochondria)",
    "metazoans_mit": "Metazoans (mitochondria)",
    "plants_mit": "Plants (mitochondria)",
    "protists_mit": "Protists (mitochondria)",
    "green_algae_plt": "Green algae (plastid)",
    "plants_plt": "Plants (plastid)",
    "protists_plt": "Protists (plastid)",
}

In [ ]:
name_map = {
    "m_both": "Polarity + length + type",
    "m_type": "Polarity + flanking-gene type",
    "m_len": "Polarity + maximum flanking-gene length",
}


def parse_kfold_block(text):
    lines = text.splitlines()
    in_block = False
    started_data = False
    out = {}

    for line in lines:
        ls = line.strip()
        if ls.startswith("--- K-fold"):
            in_block = True
            continue
        if not in_block:
            continue
        if ls == "":
            if started_data:
                break
            continue
        if ls.startswith("---"):
            break

        parts = ls.split()
        model = parts[0]
        if model not in name_map:
            continue
        if len(parts) < 3:
            continue

        elpd_diff, se_diff = parts[1], parts[2]
        out[name_map[model]] = (float(elpd_diff), float(se_diff))
        started_data = True

    if len(out) != 3:
        raise ValueError(f"Expected 3 models in K-fold block, found {len(out)}: {out}")
    return out

In [ ]:
rows_s2 = []

model_key_map = {
    "gene_length": "Polarity + maximum flanking-gene length",
    "gene_type": "Polarity + flanking-gene type",
    "both": "Polarity + length + type",
}

for g in groups:
    print(f"Processing group: {g}")
    gdir = BASE / g

    brms_pol = gdir / "brms_polarity" / "brms_polarity_row.tsv"
    brms_type_length_dir = gdir / "brms_type_length"
    brms_type_length = brms_type_length_dir / "brms_type_length_row.tsv"
    brms_type_length_txt = brms_type_length_dir / "brms_type_length.txt"

    tsv_polarity = pl.read_csv(brms_pol, separator="\t")
    tsv_type_length = pl.read_csv(brms_type_length, separator="\t")

    kfold = parse_kfold_block(brms_type_length_txt.read_text())

    best_elpd = tsv_type_length["kfold_elpd"].max()

    label = label_map[g]

    # ---- base (polarity-only) model: single row ----
    polarity = tsv_polarity.row(0, named=True)
    rows_s2.append(
        {
            "group": label,
            "model": "Polarity only",
            "N IGRs": polarity["N_regions"],
            "fold_conv": polarity["fold_convergent_over_same"],
            "fold_conv_lo": polarity["fold_convergent_lo"],
            "fold_conv_hi": polarity["fold_convergent_hi"],
            "fold_div": polarity["fold_divergent_over_same"],
            "fold_div_lo": polarity["fold_divergent_lo"],
            "fold_div_hi": polarity["fold_divergent_hi"],
            "Difference in expected log predictive density": None,
            "SE of ELPD difference": None,
        }
    )

    # ---- len / type / both models: one row each ----
    for pred_type, model_name in model_key_map.items():
        r = tsv_type_length.filter(pl.col("predictor_type") == pred_type).row(
            0, named=True
        )
        elpd_diff = r["kfold_elpd"] - best_elpd
        _, se_diff = kfold[model_name]
        rows_s2.append(
            {
                "group": label,
                "model": model_name,
                "N IGRs": r["N_regions"],
                "fold_conv": r["fold_convergent_over_same"],
                "fold_conv_lo": r["fold_convergent_lo"],
                "fold_conv_hi": r["fold_convergent_hi"],
                "fold_div": r["fold_divergent_over_same"],
                "fold_div_lo": r["fold_divergent_lo"],
                "fold_div_hi": r["fold_divergent_hi"],
                "Difference in expected log predictive density": elpd_diff,
                "SE of ELPD difference": se_diff,
            }
        )

s2 = pl.DataFrame(rows_s2)

float_cols = [col for col in s2.columns if s2[col].dtype in [pl.Float32, pl.Float64]]
s2 = s2.with_columns([pl.col(col).round(2) for col in float_cols])

s2.write_csv(BASE / "code" / "supplementary_table2.tsv", separator="\t", null_value="-")